<a href="https://colab.research.google.com/github/Nikhila1012/company-policy-rag/blob/main/Company_policy_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    transformers \
    accelerate \
    pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: transformers
    Found existing installation: transformers 5.18.0
    Uninstalling transformers-5.18.0:
      Successfully uninstalled transformers-5.18.0
  Attempting uninstall: sent

In [3]:
import warnings
warnings.filterwarnings("ignore")

import torch

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline

from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

In [16]:
# ============================================================
# CONFIGURATION
# ============================================================

PDF_PATH = "/content/AcmeTech_Employee_Policy_Manual.pdf"

CHUNK_SIZE = 700
CHUNK_OVERLAP = 100

TOP_K = 4

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5"

LLM_MODEL = "HuggingFaceTB/SmolLM2-1.7B-Instruct"

MAX_NEW_TOKENS = 100
TEMPERATURE = 0.2

In [17]:
# ============================================================
# LOAD PDF
# ============================================================

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

print(f"Pages loaded: {len(documents)}")

Pages loaded: 3


In [18]:
# ============================================================
# SPLIT DOCUMENT INTO CHUNKS
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

splits = text_splitter.split_documents(documents)

print(f"Chunks created: {len(splits)}")

Chunks created: 9


In [19]:
# ============================================================
# CREATE EMBEDDINGS
# ============================================================

embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding model loaded.


In [20]:
# ============================================================
# CREATE FAISS VECTOR DATABASE
# ============================================================

vector_db = FAISS.from_documents(
    documents=splits,
    embedding=embeddings
)

print("FAISS vector database created successfully.")

FAISS vector database created successfully.


In [21]:
# ============================================================
# CREATE RETRIEVER
# ============================================================

retriever = vector_db.as_retriever(
    search_kwargs={"k": TOP_K}
)

print(f"Retriever configured to return top {TOP_K} chunks.")

Retriever configured to return top 4 chunks.


In [22]:
# ============================================================
# LOAD LLM
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL
)

model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("LLM loaded successfully.")

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.76k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.42GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

LLM loaded successfully.


In [23]:
# ============================================================
# CREATE TEXT GENERATION PIPELINE
# ============================================================

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    do_sample=True,
    return_full_text=False
)

llm = HuggingFacePipeline(
    pipeline=hf_pipeline
)

print("Text generation pipeline created.")

Text generation pipeline created.


In [24]:
# ============================================================
# FORMAT RETRIEVED DOCUMENTS
# ============================================================

def format_docs(docs):
    cleaned = []

    for doc in docs:
        text = " ".join(doc.page_content.split())
        cleaned.append(text)

    return "\n\n".join(cleaned)

In [29]:
# ============================================================
# RAG PROMPT
# ============================================================

template = """<|system|>
You are a company policy assistant.

Use the provided context to answer the user's question.

IMPORTANT RULES:
1. If the context contains the answer, answer using that information.
2. If the context does NOT contain the answer, say exactly:
"I could not find this information in the provided policy document."
3. NEVER give both an answer and the "I could not find..." message.
4. Give ONLY ONE answer to the user's question.
5. Do not generate another question.
6. Do not generate another answer.
7. Do not continue after answering.

<|user|>
Context:
{context}

Question:
{question}

<|assistant|>
"""

prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

In [30]:
# ============================================================
# BUILD RAG CHAIN
# ============================================================

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain created successfully.")

RAG chain created successfully.


In [27]:
# ============================================================
# TEST RAG
# ============================================================

question = "What are the standard working hours?"

answer = rag_chain.invoke(question)

print("\n========== RAG ANSWER ==========\n")
print(answer.strip())

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



========== RAG ANSWER ==========

Employees normally work Monday through Friday from 9:30 AM to 6:00 PM, with a one-hour lunch break.


In [31]:
rag_chain.invoke(
    "How many casual leave days are employees entitled to?"
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Employees are entitled to 12 days of casual leave.'

In [32]:
rag_chain.invoke(
    "What is the remote work policy?"
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'The remote work policy allows employees to work remotely up to two days per week with manager approval. Employees must maintain normal working hours and remain reachable through approved communication tools.'

In [33]:
# -- testing hallucination resistance --
rag_chain.invoke(
    "What is AcmeTech's annual revenue?"
)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'I could not find this information in the provided policy document.'